# Classify family units in the AFC data

create parent_household_age (potential parent ID, age, household ID) and child_party (potential child ID, household ID, age, and whether they live in a fully Republican, fully Democratic, or mixed Democratic/Republican household.

In [ ]:
import pandas as pd
import zipfile
import gzip
import datetime
import numpy as np

In [ ]:
with gzip.open('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz') as f:
    patient_baseline = pd.read_csv(f)

In [ ]:
len(patient_baseline)

In [ ]:
patients_household = patient_baseline[~patient_baseline['household_id'].isna()]
patients_household = patients_household[['patientuid', 'gender', 'maritalstatus', 
       'dob','household_id']]

In [ ]:
len(patients_household)

Proportion of patients with households

In [ ]:
len(patients_household)/len(patient_baseline)

In [ ]:
def calculate_age(born):
    today = datetime.date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

In [ ]:
patients_household.loc[:,'dob'] = pd.to_datetime(patients_household['dob'], errors='coerce')

In [ ]:
patients_household.loc[:,'age'] = patients_household['dob'].apply(calculate_age)

In [ ]:
def find_parent_child_pairs_by_household(df):
    parent_child_pairs = []

    # group by household_id to check each household
    for _, household_df in df.groupby('household_id'):
        # all pairs within the same household
        for i, parent_row in household_df.iterrows():
            for j, child_row in household_df.iterrows():
                # parent is older than the child
                if parent_row['age'] > child_row['age']:
                    age_diff = parent_row['age'] - child_row['age']
                    # age difference is between 12 and 55 years
                    if 12 <= age_diff <= 55:
                        parent_child_pairs.append((parent_row['age'], parent_row['patientuid'], child_row['age'], child_row['patientuid'], parent_row['household_id']))

    # Convert the list of pairs to a DataFrame
    pairs_df = pd.DataFrame(parent_child_pairs, columns=['parent_age', 'parent_id', 'child_age', 'child_id', 'household_id'])
    return pairs_df

In [ ]:
parent_child_df = find_parent_child_pairs_by_household(patients_household)

In [ ]:
len(parent_child_df['household_id'].unique()), len(parent_child_df['household_id'].unique())/len(parent_child_df['household_id'])

In [ ]:
len(parent_child_df['parent_id'].unique()), len(parent_child_df['child_id'].unique())

In [ ]:
# save for emergencies
old_parent_child_df = parent_child_df.copy()

In [ ]:
# add parent count -- number of matched parents in L2
parent_count = parent_child_df.groupby('child_id')['parent_id'].nunique().reset_index()
parent_count.columns = ['child_id', 'unique_parent_count']

In [ ]:
save_filename = '/share/pi/deho-pi/AFC/mortonc/intermediate/parent_count.csv'
np.save(save_filename, parent_count) 

## Link up with the L2 data

In [ ]:
zf = zipfile.ZipFile('/share/pi/deho-pi/AFC/l2_afc_match_032125.csv.zip') 
l2_afc = pd.read_csv(zf.open('l2_afc_match_032125.csv'))

In [ ]:
# parents with known political affiliation
known_parents = parent_child_df[parent_child_df['parent_id'].isin(l2_afc['patientuid'])]
print(len(known_parents['parent_id'].unique()), len(known_parents['parent_id'].unique())/len(parent_child_df['parent_id'].unique()))

In [ ]:
len(parent_child_df['child_id'].unique())

In [ ]:
# restrict to households where ate least one parent's political leanings are Democrat or Republican
dem_rep_ids = l2_afc[l2_afc['Parties_Description'].isin(['Republican', 'Democratic'])]['patientuid']
print(len(dem_rep_ids))

known_parents = parent_child_df[parent_child_df['parent_id'].isin(dem_rep_ids)]
print(len(known_parents['parent_id'].unique()), len(known_parents['parent_id'].unique())/len(parent_child_df['parent_id'].unique()))

known_children = parent_child_df[parent_child_df['child_id'].isin(dem_rep_ids)]
print(len(known_children['child_id'].unique()), len(known_children['child_id'].unique())/len(parent_child_df['child_id'].unique()))

In [ ]:
dem_parents = parent_child_df[parent_child_df['parent_id'].isin(l2_afc[l2_afc['Parties_Description'] == 'Democratic']['patientuid'])]
print(len(dem_parents['parent_id'].unique()), len(dem_parents['parent_id'].unique())/len(parent_child_df['parent_id'].unique()))

rep_parents = parent_child_df[parent_child_df['parent_id'].isin(l2_afc[l2_afc['Parties_Description'] == 'Republican']['patientuid'])]
print(len(rep_parents['parent_id'].unique()), len(rep_parents['parent_id'].unique())/len(parent_child_df['parent_id'].unique()))


3 million adults have known political party

44% (200,000) potential parents are recorded Democrats or Republicans

25% (120,000) potential children are recorded Democrats or Republicans

## Fill in all potential children based on household_id. 
All potential children (including adult children) receive Democrat if all potential parents in their household are Democrats, Republican if all Republican, and 'neither' if there is at least one Democrat and at least one Republican in their household.

In [ ]:
# get whether household is Democratic, Republican, or neither
# record parent party
parent_parties = l2_afc[['patientuid', 'Parties_Description']]
parent_parties = parent_parties.rename(columns={'patientuid': 'parent_id'})

known_parents = pd.merge(known_parents, parent_parties)

#rep = l2_afc[l2_afc['Parties_Description'].isin(['Republican', 'Democratic'])]['patientuid']

In [ ]:
household_party = known_parents.groupby(['household_id', 'Parties_Description'])['Parties_Description'].count()
household_party.index = household_party.index.rename({'Parties_Description': 'party'})
household_party = household_party.reset_index()

In [ ]:
len(household_party['household_id'].unique()), len(household_party['household_id'].unique())/len(household_party)

97% of households in the data (170,000) are either solely Democrat or solely Republican in terms of the adults in the household.

In [ ]:
# household ids of undetermined households
household_party_undetermined = household_party.groupby('household_id')['party'].count()>1
household_party_undetermined = household_party_undetermined[household_party_undetermined].index

household_party_rep = household_party[household_party['party'] == 'Republican'] 
household_party_rep = household_party_rep[~household_party_rep['household_id'].isin(household_party_undetermined)]['household_id']

household_party_dem = household_party[household_party['party'] == 'Democratic'] 
household_party_dem = household_party_dem[~household_party_dem['household_id'].isin(household_party_undetermined)]['household_id']

In [ ]:
dem_children = known_parents[known_parents['household_id'].isin(household_party_dem)]['child_id'].unique()
rep_children = known_parents[known_parents['household_id'].isin(household_party_rep)]['child_id'].unique()
unk_children = known_parents[known_parents['household_id'].isin(household_party_undetermined)]['child_id'].unique()

In [ ]:
print(len(dem_children), len(rep_children), len(unk_children))

## Export dataset of parent ID, age, household ID

In [ ]:
# we want a dataset with IDs of potential parents, household ID, and age

In [ ]:
parent_df = parent_child_df[['parent_id', 'parent_age', 'household_id']].drop_duplicates()

In [ ]:
# write 
save_filename = '/share/pi/deho-pi/AFC/parent_household_age.npy'
np.save(save_filename, parent_df) 

## Export dataset of child ID, age, household ID, household politics

In [ ]:
# create dataframe with household political affiliation, if known, and potential children's patient ID
dem_children_df = pd.DataFrame({'child_ID':dem_children, 'household_party': 'Democratic'})
rep_children_df = pd.DataFrame({'child_ID':rep_children, 'household_party': 'Republican'})
unk_children_df = pd.DataFrame({'child_ID':unk_children, 'household_party': 'Undetermined'})
party_children_df = pd.concat([dem_children_df, rep_children_df, unk_children_df])

In [ ]:
# merge with all potential children, where household political affiliation is NA if unknown
child_df = parent_child_df[['child_id', 'child_age', 'household_id']].drop_duplicates()


In [ ]:
child_df = pd.merge(child_df, party_children_df, left_on = 'child_id', right_on = 'child_ID', how = "left")[['child_id', 'child_age', 'household_id', 'household_party']]

In [ ]:
# write 
save_filename = '/share/pi/deho-pi/AFC/child_party.npy'
np.save(save_filename, child_df) 